In [ ]:
import sys
from pathlib import Path

import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loader import careful_load_csv

dataset_path = ROOT / "data" / "catapulta_doe.csv"
df = careful_load_csv(dataset_path)

features = [
    "release_angle",
    "firing_angle",
    "cup_elevation",
    "pin_elevation",
    "bungee_elevation",
]
target = "distancia"

X = df[features]
y = df[target]

modelo = Pipeline(
    [
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("reg", LinearRegression()),
    ]
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
resultados_r2 = []
resultados_rmse = []

for fold, (train_index, test_index) in enumerate(kf.split(X), start=1):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    resultados_r2.append(r2)
    resultados_rmse.append(rmse)

    print(f"Fold {fold} | R²: {r2:.4f} | RMSE: {rmse:.4f}")

print("-" * 30)
print("MÉDIA FINAL 5-FOLD:")
print(f"R² Médio: {np.mean(resultados_r2):.4f}")
print(f"RMSE Médio: {np.mean(resultados_rmse):.4f}")
print(f"Amostras avaliadas: {len(df)}")